# Retrain extractor v2 — Qwen3-1.7B + LoRA (feature 006)

Khác bản v1:

| | v1 | v2 |
|---|---|---|
| Prompt | 13 luật + schema (~1.300 token), khác prompt lúc serve | `app/services/prompt_v2.py` (~430 token), **dùng chung cho train và serve** |
| Loss | tính trên cả prompt | **chỉ trên JSON trả lời** (`train_on_responses_only`) |
| Mẫu bị cắt | 44/395 vượt 4.096 token (mất đuôi certificates/honours/languages) | 0 (dài nhất ~3.700) |
| Dữ liệu | 395 CV | 412 CV + 412 bản đổi tên công ty/trường (augment), dev 30 CV riêng |
| JSON đích | có khoảng trắng | gọn (`,` `:`), ít hơn ~12% token sinh ra |

**v4 (feature 013, mặc định):** `python -m training.build_dataset_v4` → upload `data/retrain_v4/` lên `MyDrive/cv-analysis-service/retrain_v4/`; `RUN_NAME = "v4"` (prompt v3, text tách theo cột). Bảng cuối so với v3 trên 20 CV thật.

**v3 (feature 007):** `python -m training.build_dataset_v3` → upload `data/retrain_v3/` lên Drive thành `MyDrive/cv-analysis-service/retrain_v3/`; `RUN_NAME = "v3"`. Notebook chấm thêm trên 20 CV thật (`test_real`) và in bảng so với v2.

**Tiết kiệm GPU:** `EVAL = "real"` (mặc định) chỉ chấm 20 CV thật (~5 phút) thay vì cả val/val_swap (~1 giờ, đặt `"full"` khi cần kiểm tra hồi quy). `SERVE = True`: sau khi xuất GGUF, cell [10] chạy API + Cloudflare tunnel và in URL — gửi URL cho Claude để test qua API (đúng cách serve thật), cell [11] giữ session sống.

**Cách chạy**
1. Trên máy local (thư mục `ai-service/`): `python -m training.build_dataset` → upload thư mục `data/retrain_v2/` lên Drive thành `MyDrive/cv-analysis-service/retrain_v2/` (biến thể D: `--hints` → `retrain_v2_hints/`, đặt `RUN_NAME = "v2_hints"`).
2. Runtime → Change runtime type → **T4 GPU**.
3. (Khuyên dùng) chạy thử nhanh với `SMOKE = True` (~10 phút), rồi đặt lại `False` và **Runtime → Run all** (~1–1,5 giờ: train ~35–50 phút, đánh giá ~30 phút, xuất GGUF ~10 phút).
4. Kết quả trên Drive `finetune_output_v2/`: `lora_adapter/`, `eval_run/report.md`, `gguf/*.gguf`. Gửi lại `eval_run/report.json` (hoặc chép cả `eval_run/`) cho Claude để so với v1.

Nếu Colab ngắt giữa chừng: chạy lại cell 1–3, rồi cell **"Nạp lại adapter"** và tiếp tục từ phần đánh giá.

In [ ]:
# [1] Cấu hình
RUN_NAME   = "v4"          # "v4" (feature 013, text tách theo cột), "v3" (feature 007), "v2" hoặc "v2_hints"
SMOKE      = False         # True: train 10 bước + đánh giá 8 CV, để kiểm tra pipeline
DRIVE_ROOT = "/content/drive/MyDrive/cv-analysis-service"
DATA_DIR   = f"{DRIVE_ROOT}/retrain_{RUN_NAME}"
OUT_DIR    = f"{DRIVE_ROOT}/finetune_output_{RUN_NAME}" + ("_smoke" if SMOKE else "")
REPO, BRANCH = "https://github.com/long20102004/recruit_system.git", "hocnv/DO-06"

BASE_MODEL = "unsloth/Qwen3-1.7B"
MAX_LEN    = 4096          # dài nhất trong dataset ~3.700 token: không mẫu nào bị cắt
EPOCHS     = 2             # v4: 895 mẫu x 2 epoch ≈ 224 bước (batch hiệu dụng 8)
LR         = 2e-4
LORA_R     = 16
EVAL_BATCH = 8             # số CV sinh song song khi đánh giá (giảm nếu hết VRAM)
MAX_NEW_TOKENS = 2048      # JSON dài nhất ~1.600 token
EVAL  = "real"            # "real": chỉ 20 CV thật (~5 phút) | "full": thêm val + val_swap 198 CV (~1 giờ) | "none": bỏ qua
SERVE = True              # sau khi xuất GGUF: chạy API + tunnel để Claude test trực tiếp (cell [10])

In [ ]:
# [2] Code đúng branch + thư viện
%cd /content
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
!pip install -q unsloth rapidfuzz json-repair
%cd /content/recruit_system/ai-service
!git log --oneline -1
import sys
sys.path.insert(0, "/content/recruit_system/ai-service")

In [ ]:
# [3] Dữ liệu từ Drive + kiểm tra prompt trong dataset == prompt serving của branch
from unsloth import FastLanguageModel   # import trước transformers/trl
import json, os, torch
from google.colab import drive
drive.mount("/content/drive")
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

def load(name):
    with open(f"{DATA_DIR}/{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

train, dev, val, val_swap = (load(n) for n in ("train", "dev", "val", "val_swap"))
test_real = load("test_real") if os.path.exists(f"{DATA_DIR}/test_real.jsonl") else []
summary = json.load(open(f"{DATA_DIR}/summary.json"))
print({"train": len(train), "dev": len(dev), "val": len(val), "val_swap": len(val_swap),
       "test_real": len(test_real), "hints": summary["hints"]})

import importlib
PROMPT_VERSION = "v3" if RUN_NAME.startswith(("v3", "v4")) else "v2"   # v4 dùng prompt v3
render_prompt = importlib.import_module(f"app.services.prompt_{PROMPT_VERSION}").render_prompt
if not summary["hints"]:
    bad = sum(r["prompt"] != render_prompt(r["input"]) for r in val + dev + train[:100])
    assert bad == 0, f"{bad} prompt khác app/services/prompt_{PROMPT_VERSION}.py của branch: build lại dataset"
    print("Prompt dataset == prompt serving: OK")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# [4] Model gốc 4-bit + LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

# Không mẫu nào được vượt MAX_LEN (v1 bị cắt 11% mẫu)
lens = [len(tokenizer(r["prompt"] + r["completion"], add_special_tokens=False).input_ids) for r in train + dev]
print(f"token/mẫu: max={max(lens)}, trung bình={sum(lens)/len(lens):.0f}, vượt {MAX_LEN}: {sum(l > MAX_LEN for l in lens)}")
assert max(lens) <= MAX_LEN

In [ ]:
# [5] Trainer: loss CHỈ tính trên phần JSON trả lời
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def to_ds(rows):
    return Dataset.from_list([{"text": r["prompt"] + r["completion"]} for r in rows])

train_ds, dev_ds = to_ds(train), to_ds(dev)
if SMOKE:
    train_ds, dev_ds = train_ds.select(range(40)), dev_ds.select(range(6))

cfg = dict(
    dataset_text_field="text", packing=False, dataset_num_proc=2,
    per_device_train_batch_size=2, gradient_accumulation_steps=4, per_device_eval_batch_size=2,
    num_train_epochs=EPOCHS, max_steps=10 if SMOKE else -1,
    learning_rate=LR, warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01, optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5, eval_strategy="steps", eval_steps=5 if SMOKE else 25,
    save_strategy="steps", save_steps=5 if SMOKE else 25, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    output_dir="/content/ckpt", report_to="none", seed=42,
)
try:
    args = SFTConfig(max_length=MAX_LEN, **cfg)          # TRL mới
except TypeError:
    args = SFTConfig(max_seq_length=MAX_LEN, **cfg)      # TRL cũ
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=train_ds, eval_dataset=dev_ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=dev_ds, args=args)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n<think>\n\n</think>\n\n",
)

# Kiểm tra mask: phần được tính loss phải bắt đầu bằng '{' và không chứa prompt
labels = trainer.train_dataset[0]["labels"]
kept = [t for t in labels if t != -100]
txt = tokenizer.decode(kept)
print("tính loss trên", len(kept), "/", len(labels), "token:", repr(txt[:60]), "...", repr(txt[-20:]))
assert txt.lstrip().startswith("{") and "<|im_start|>" not in txt, "Mask sai: loss vẫn tính trên prompt"

In [ ]:
# [6] Train (~35–50 phút trên T4) và lưu adapter tốt nhất (theo dev loss) lên Drive
stats = trainer.train()
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
print("train loss (bước, loss):", tr[::max(1, len(tr) // 10)])
print("dev loss:", ev)
print("checkpoint tốt nhất:", trainer.state.best_model_checkpoint)

ADAPTER_DIR = f"{OUT_DIR}/lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
json.dump({"train_loss": tr, "dev_loss": ev, "best": trainer.state.best_model_checkpoint},
          open(f"{OUT_DIR}/train_log.json", "w"))
print("Đã lưu", ADAPTER_DIR)
# Model mới: dự đoán cũ (nếu train lại cùng RUN_NAME) không còn đúng, xoá để cell [7] chạy lại từ đầu
old_pred = f"{OUT_DIR}/eval_run/predictions.jsonl"
if os.path.exists(old_pred):
    os.remove(old_pred); print("Đã xoá dự đoán của lần train trước")

### Nạp lại adapter (chỉ khi Colab đã ngắt sau bước train)
Chạy cell 1–3, bỏ comment và chạy cell dưới (bỏ qua cell [4]–[6]), rồi chạy tiếp từ cell [7]. Cell [7] tự bỏ qua các CV đã có trong `eval_run/predictions.jsonl` trên Drive.

In [ ]:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=f"{OUT_DIR}/lora_adapter", max_seq_length=MAX_LEN, dtype=None, load_in_4bit=True)

In [ ]:
# [7] Sinh dự đoán (theo EVAL: 20 CV thật, hoặc thêm val + val_swap), lưu đúng format eval.predict
import time
from app.utils.json_parse import extract_json

FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
STOP_IDS = tokenizer.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"])

RUN_DIR = f"{OUT_DIR}/eval_run"
os.makedirs(RUN_DIR, exist_ok=True)
PRED_PATH = f"{RUN_DIR}/predictions.jsonl"

def predict(rows):
    done = set()
    if os.path.exists(PRED_PATH):
        done = {json.loads(l)["id"] for l in open(PRED_PATH, encoding="utf-8")}
    todo = sorted((r for r in rows if r["id"] not in done), key=lambda r: len(r["prompt"]))
    with open(PRED_PATH, "a", encoding="utf-8") as f:
        for i in range(0, len(todo), EVAL_BATCH):
            batch = todo[i:i + EVAL_BATCH]
            enc = tokenizer([r["prompt"] for r in batch], return_tensors="pt", padding=True,
                            add_special_tokens=False).to("cuda")
            t0 = time.time()
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True,
                                     eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id)
            dt = time.time() - t0
            for r, seq in zip(batch, out[:, enc.input_ids.shape[1]:].tolist()):
                n_out = next((k for k, t in enumerate(seq) if t in STOP_IDS), len(seq))
                text = tokenizer.decode(seq[:n_out], skip_special_tokens=True).strip()
                try:
                    data, status = extract_json(text), 200
                except Exception:
                    data, status = None, 500
                f.write(json.dumps({"id": r["id"], "status": status, "latency_s": round(dt / len(batch), 3),
                                    "output_tokens": n_out, "hit_cap": n_out >= MAX_NEW_TOKENS, "data": data},
                                   ensure_ascii=False) + "\n")
            f.flush()
            print(f"{min(i + EVAL_BATCH, len(todo))}/{len(todo)} | {dt:.0f}s/batch", flush=True)

# 20 CV thật chạy trước (quan trọng nhất); nếu Colab ngắt, chạy lại cell này sẽ bỏ qua CV đã có trong predictions.jsonl
groups = {"none": [], "real": [test_real], "full": [test_real, val + val_swap]}[EVAL]
if SMOKE:
    groups = [test_real[:2]] + ([val[:4] + val_swap[:4]] if EVAL == "full" else [])
for rows in groups:
    predict(rows)
print("EVAL =", EVAL, "| đã dự đoán:", sum(len(g) for g in groups), "CV")

In [ ]:
# [8] Chấm điểm bằng đúng harness của feature 003 (A thô, B +hậu kiểm, C +KG) và so với v1
if EVAL == "none":
    print("EVAL = none: bỏ qua chấm điểm")
else:
    from eval.report import build_report
    splits = ([f"{DATA_DIR}/test_real.jsonl"] if test_real else []) + ([f"{DATA_DIR}/val.jsonl", f"{DATA_DIR}/val_swap.jsonl"] if EVAL == "full" else [])
    if SMOKE:
        for name, rows in (("val", val[:4] if EVAL == "full" else []), ("val_swap", val_swap[:4] if EVAL == "full" else []), ("test_real", test_real[:2])):
            if not rows:
                continue
            with open(f"{RUN_DIR}/{name}.jsonl", "w", encoding="utf-8") as f:
                f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
        splits = [f"{RUN_DIR}/{n}.jsonl" for n in ("val", "val_swap", "test_real") if os.path.exists(f"{RUN_DIR}/{n}.jsonl")]
    build_report(RUN_DIR, splits, variants=("A", "B", "C"))

    V1 = {"val": {"experience.company": 0.918, "education.school": 0.961, "skills": 0.839, "certificates": 0.89, "honours": 0.768, "languages.language": 0.849, "hallucination_rate": 0.0017}, "val_swap": {"experience.company": 0.926, "education.school": 0.966, "skills": 0.829, "certificates": 0.824, "honours": 0.693, "languages.language": 0.799, "hallucination_rate": 0.0018}}   # feature 003, model v1, biến thể B, Colab T4
    rep = json.load(open(f"{RUN_DIR}/report.json"))
    preds = [json.loads(l) for l in open(PRED_PATH, encoding="utf-8")]
    print(f"JSON hợp lệ: {sum(p['status'] == 200 for p in preds)}/{len(preds)} | chạm trần token: {sum(p['hit_cap'] for p in preds)}"
          f" | token sinh ra TB: {sum(p['output_tokens'] for p in preds) / len(preds):.0f}")
    if "val" in rep["splits"]:
        print(f"{'trường (biến thể B)':28}{'v1 val':>8}{'v2 val':>8}{'v1 swap':>9}{'v2 swap':>9}")
        for field in V1["val"]:
            row = []
            for sp in ("val", "val_swap"):
                m = rep["splits"][sp]["B"]["metrics"]
                v2 = m["hallucination_rate"] if field == "hallucination_rate" else m["fields"][field]["f1"]
                row += [V1[sp][field], v2]
            print(f"{field:28}{row[0]:8.3f}{row[1]:8.3f}{row[2]:9.3f}{row[3]:9.3f}")
    if "test_real" in rep["splits"]:
        V2_REAL = {"experience.company": 0.780, "education.school": 1.0, "personal_projects.name": 0.860, "skills": 0.822, "skills_canonical": 0.832, "certificates": 0.556, "honours": 0.0, "languages.language": 0.759}   # model v3 trên text tách theo cột, 19 CV thật (feature 011), biến thể B
        print("\n20 CV thật (test_real), biến thể B:")
        print(f"{'trường':28}{'v3':>8}{'mới':>8}")
        m = rep["splits"]["test_real"]["B"]["metrics"]["fields"]
        for field, base in V2_REAL.items():
            print(f"{field:28}{base:8.3f}{(m[field]['f1'] or 0):8.3f}")
    print("\nBáo cáo đầy đủ:", f"{RUN_DIR}/report.md")

In [ ]:
# [9] Xuất GGUF Q4_K_M (file duy nhất cần để serve bằng llama.cpp)
model.save_pretrained_gguf(f"{OUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
import glob
for p in glob.glob(f"{OUT_DIR}/gguf/**/*.gguf", recursive=True) + glob.glob(f"{OUT_DIR}/gguf*.gguf"):
    print(p, round(os.path.getsize(p) / 1e9, 2), "GB")
print(f"Serve: đặt file vào ai-service/merged_model/model-{RUN_NAME}-q4_k_m.gguf, chạy với PROMPT_VERSION={RUN_NAME}")

In [ ]:
# [10] Chạy API bằng file GGUF vừa xuất + mở tunnel, để Claude test trực tiếp (SERVE = True)
# Giải phóng GPU của model train, cài llama.cpp CUDA, chạy đúng code serve của branch với PROMPT_VERSION của lần train.
if SERVE:
    import gc, glob, subprocess, time, re, urllib.request, site
    for _n in ("model", "trainer"):
        globals().pop(_n, None)
    gc.collect(); torch.cuda.empty_cache()
    !pip install -q fastapi uvicorn pydantic json-repair rapidfuzz
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
    open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
    !ldconfig

    gguf = sorted(glob.glob(f"{OUT_DIR}/gguf*/**/*.gguf", recursive=True) + glob.glob(f"{OUT_DIR}/gguf*.gguf"))
    assert gguf, "Chưa có file GGUF: chạy cell [9] trước"
    GGUF_PATH = next((g for g in gguf if "q4_k_m" in g.lower()), gguf[0])
    print("Model:", GGUF_PATH)
    %cd /content/recruit_system/ai-service
    env = dict(os.environ, PROMPT_VERSION=PROMPT_VERSION, GGUF_MODEL_PATH=GGUF_PATH,
               LD_LIBRARY_PATH=":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
    env.pop("KB_DSN", None)   # Colab không có Postgres: KB từ seed CSV
    srv = subprocess.Popen(["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
                           stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=env)
    h = None
    for _ in range(150):
        try:
            h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")); break
        except Exception:
            time.sleep(2)
    print("health:", h)
    assert h and h["engine"] == "llama_cpp" and h["prompt_version"] == PROMPT_VERSION, "Service chưa chạy đúng: !tail -50 server.log"

    tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                           stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
    url = None
    for _ in range(30):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
        if m:
            url = m.group(0); break
    print("=" * 70)
    print("URL ai-service (gửi cho Claude):", url)
    print("=" * 70)


In [ ]:
# [11] Giữ session sống trong lúc Claude test (dừng cell này khi xong)
while SERVE:
    log = open("/content/recruit_system/ai-service/server.log").read()
    times = [float(x) for x in re.findall(r"Inference time: ([0-9.]+)s", log)]
    avg = f"{sum(times) / len(times):.1f}s" if times else "-"
    print(time.strftime("%H:%M:%S"), "| requests:", len(times), "| trung bình:", avg,
          "| service sống:", srv.poll() is None, flush=True)
    time.sleep(60)
